# 04 FeatureEngineering

Notebook นี้ใช้สำหรับสร้างฟีเจอร์จากข้อมูล timestamp ที่ผ่านการ clean แล้ว


In [69]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')


## Load Clean Data

โหลดไฟล์ `vw_timestamp_dashboard_clean.csv` ซึ่งเป็นข้อมูลที่ผ่านการ clean แล้วมาใช้งาน


In [70]:
source_path = '../../data/interim/vw_timestamp_dashboard_clean.csv'
df = pd.read_csv(source_path, encoding='utf-8-sig')

print(f'source: {source_path}')
print(f'shape: {df.shape}')
df.head()


source: ../../data/interim/vw_timestamp_dashboard_clean.csv
shape: (26538, 37)


,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,PRESTIGEFittingSapAmount,NEUSTILEFittingSapAmount,DURAFittingSapAmount,ACCESSORIESSapAmount,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd
0,SB1,0.0,2025-01-02 07:18:56,1.0,3.0,71-4711,SB1PL250102001,"SCGR Cholburi Plant SCG Roofing Co., Ltd.",2025-01-02 07:19:02,0.0,...,0.0,0.0,0.0,0.0,2025-01-02 08:30:33,2025-01-02 08:30:33,NaN,NaN,NaN,NaN
1,SB1,0.0,2025-01-02 07:21:43,2.0,3.0,71-3545,SB1PL250102002,"SCGR Cholburi Plant SCG Roofing Co., Ltd.",2025-01-02 07:21:46,0.0,...,0.0,0.0,0.0,0.0,2025-01-02 08:22:25,2025-01-02 08:32:32,NaN,NaN,NaN,NaN
2,SB1,0.0,2025-01-02 10:52:33,3.0,1.0,89-1359,SB1PL250102004,หจก.สำรวยเซรามิค,2025-01-02 10:52:39,0.0,...,0.0,0.0,0.0,0.0,2025-01-02 11:21:54,2025-01-02 11:23:32,2025-01-02 10:57:19,2025-01-02 10:59:32,NaN,NaN
3,SB1,0.0,2025-01-02 12:05:56,4.0,3.0,71-6663,SB1PL250102009,"SCGR Lumpoon Plant SCG Roofing Co., Ltd.",2025-01-02 12:05:59,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,2025-01-02 12:13:03,2025-01-02 12:14:35,NaN,NaN
4,SB1,0.0,2025-01-02 13:12:26,5.0,1.0,70-6399,SB1PL250102010,กรุงเทพฯ ดอนเมือง,2025-01-02 13:12:30,0.0,...,0.0,0.0,0.0,0.0,2025-01-02 13:24:44,2025-01-02 13:25:22,NaN,NaN,NaN,NaN


## Convert Timestamp Columns

แปลงคอลัมน์วันที่และเวลาที่เกี่ยวข้องให้เป็นชนิด `datetime` เพื่อให้สามารถคำนวณต่อได้ถูกต้อง


In [71]:
timestamp_columns = [
    'PickDate',
    'QueueTime',
    'OperatorCarConfirm',
    'CarConfirm',
    'FirstPostPallet',
    'LastPostPallet',
    'PostingTime',
    'TileStart',
    'TileEnd',
    'FittingStart',
    'FittingEnd',
    'AccStart',
    'AccEnd',
]

df_feature = df.copy()

for col in timestamp_columns:
    df_feature[col] = pd.to_datetime(df_feature[col], errors='coerce')

df_feature[timestamp_columns].dtypes

PickDate              datetime64[ns]
QueueTime             datetime64[ns]
OperatorCarConfirm    datetime64[ns]
CarConfirm            datetime64[ns]
FirstPostPallet       datetime64[ns]
LastPostPallet        datetime64[ns]
PostingTime           datetime64[ns]
TileStart             datetime64[ns]
TileEnd               datetime64[ns]
FittingStart          datetime64[ns]
FittingEnd            datetime64[ns]
AccStart              datetime64[ns]
AccEnd                datetime64[ns]
dtype: object

## Target Variable: `total_time`

คำนวณเวลารวมของกระบวนการจัดส่ง (หน่วย: นาที)

```
total_time = PostingTime - OperatorCarConfirm
```

- `OperatorCarConfirm` = เวลาที่ Operator ยืนยันรถเข้าคิว (จุดเริ่มต้น)
- `PostingTime` = เวลาที่ Post สำเร็จ (จุดสิ้นสุด)


In [72]:
df_feature['total_time_min'] = (
    (df_feature['PostingTime'] - df_feature['OperatorCarConfirm'])
    .dt.total_seconds() / 60
).round(2)

print(f'total_time_min null: {df_feature["total_time_min"].isna().sum()}')
print(f'total_time_min < 0 : {(df_feature["total_time_min"] < 0).sum()}')
df_feature[['OperatorCarConfirm', 'PostingTime', 'total_time_min']].head(10)


total_time_min null: 0
total_time_min < 0 : 0


,OperatorCarConfirm,PostingTime,total_time_min
0,2025-01-02 07:18:57,2025-01-02 08:31:00,72.05
1,2025-01-02 07:21:45,2025-01-02 08:43:43,81.97
2,2025-01-02 10:52:36,2025-01-02 11:32:59,40.38
3,2025-01-02 12:05:57,2025-01-02 12:43:57,38.00
4,2025-01-02 13:12:27,2025-01-02 13:41:07,28.67
5,2025-01-02 13:13:09,2025-01-02 13:58:21,45.20
6,2025-01-02 13:16:44,2025-01-02 13:56:48,40.07
7,2025-01-02 13:39:11,2025-01-02 14:37:57,58.77
8,2025-01-02 13:40:04,2025-01-02 14:15:17,35.22
9,2025-01-02 14:22:33,2025-01-02 15:10:53,48.33


## Calendar Features จาก `OperatorCarConfirm`

แตกวันที่/เวลาออกเป็นฟีเจอร์ย่อย:
- `hour` — ชั่วโมง
- `day_of_week` — วันในสัปดาห์ (0=จันทร์, 6=อาทิตย์)
- `week_of_month` — สัปดาห์ที่เท่าไรของเดือน
- `month` — เดือน


In [73]:
occ = df_feature['OperatorCarConfirm']

df_feature['hour']         = occ.dt.hour
df_feature['day_of_week']  = occ.dt.dayofweek
df_feature['week_of_month'] = ((occ.dt.day - 1) // 7 + 1).astype('Int64')
df_feature['month']        = occ.dt.month
df_feature['year']         = occ.dt.year

calendar_cols = ['OperatorCarConfirm', 'hour', 'day_of_week', 'week_of_month', 'month', 'year']
df_feature[calendar_cols].head(10)


,OperatorCarConfirm,hour,day_of_week,week_of_month,month,year
0,2025-01-02 07:18:57,7,3,1,1,2025
1,2025-01-02 07:21:45,7,3,1,1,2025
2,2025-01-02 10:52:36,10,3,1,1,2025
3,2025-01-02 12:05:57,12,3,1,1,2025
4,2025-01-02 13:12:27,13,3,1,1,2025
5,2025-01-02 13:13:09,13,3,1,1,2025
6,2025-01-02 13:16:44,13,3,1,1,2025
7,2025-01-02 13:39:11,13,3,1,1,2025
8,2025-01-02 13:40:04,13,3,1,1,2025
9,2025-01-02 14:22:33,14,3,1,1,2025


## Product Features — ปริมาณสินค้า

รวมยอด SAP amount แยกตามกลุ่มสินค้า แล้วสร้าง flag และนับจำนวนกลุ่มที่มี:
- `total_tile_amount` / `total_fitting_amount` / `total_accessories_amount` — ยอดแต่ละกลุ่ม
- `total_sap_amount` — ยอดรวมทั้งหมด
- `has_tile` / `has_fitting` / `has_accessories` — 1 ถ้ามีสินค้ากลุ่มนั้น
- `product_group_count` — จำนวนกลุ่มสินค้าที่ order นี้มี


In [74]:
amount_columns = [
    'CPACTileSapAmount',
    'PRESTIGETileSapAmount',
    'NEUSTILETileSapAmount',
    'CPACFittingSapAmount',
    'PRESTIGEFittingSapAmount',
    'NEUSTILEFittingSapAmount',
    'DURAFittingSapAmount',
    'ACCESSORIESSapAmount',
]

tile_columns    = ['CPACTileSapAmount', 'PRESTIGETileSapAmount', 'NEUSTILETileSapAmount']
fitting_columns = ['CPACFittingSapAmount', 'PRESTIGEFittingSapAmount', 'NEUSTILEFittingSapAmount', 'DURAFittingSapAmount']

df_feature[amount_columns] = df_feature[amount_columns].apply(pd.to_numeric, errors='coerce').fillna(0)

df_feature['total_tile_amount']        = df_feature[tile_columns].sum(axis=1)
df_feature['total_fitting_amount']     = df_feature[fitting_columns].sum(axis=1)
df_feature['total_accessories_amount'] = df_feature['ACCESSORIESSapAmount']
df_feature['total_sap_amount']         = df_feature[amount_columns].sum(axis=1)

df_feature['has_tile']        = (df_feature['total_tile_amount'] > 0).astype(int)
df_feature['has_fitting']     = (df_feature['total_fitting_amount'] > 0).astype(int)
df_feature['has_accessories'] = (df_feature['total_accessories_amount'] > 0).astype(int)
df_feature['product_group_count'] = df_feature[['has_tile', 'has_fitting', 'has_accessories']].sum(axis=1)

product_cols = [
    'total_tile_amount', 'total_fitting_amount', 'total_accessories_amount', 'total_sap_amount',
    'has_tile', 'has_fitting', 'has_accessories', 'product_group_count',
]
df_feature[product_cols].head(10)


,total_tile_amount,total_fitting_amount,total_accessories_amount,total_sap_amount,has_tile,has_fitting,has_accessories,product_group_count
0,7680.0,0.0,0.0,7680.0,1,0,0,1
1,6720.0,0.0,0.0,6720.0,1,0,0,1
2,1500.0,118.0,0.0,1618.0,1,1,0,2
3,0.0,523.0,0.0,523.0,0,1,0,1
4,1300.0,0.0,0.0,1300.0,1,0,0,1
5,1450.0,145.0,0.0,1595.0,1,1,0,2
6,968.0,0.0,95.0,1063.0,1,0,1,2
7,2720.0,233.0,47.0,3000.0,1,1,1,3
8,240.0,0.0,0.0,240.0,1,0,0,1
9,1000.0,282.0,0.0,1282.0,1,1,0,2


## Phase Time Features (นาที)

แตก `total_time` ออกเป็น 4 ช่วงย่อยตาม pipeline การจัดส่ง:

| Feature | ช่วงเวลา | ความหมาย |
|---|---|---|
| `wait_call_min` | `OperatorCarConfirm` → `CarConfirm` | รอรถเข้าช่อง |
| `prepare_loading_min` | `CarConfirm` → `FirstPostPallet` | เตรียมสินค้า |
| `loading_time_min` | `FirstPostPallet` → `LastPostPallet` | โหลดสินค้า |
| `close_job_min` | `LastPostPallet` → `PostingTime` | ปิดงาน / Posting |


In [75]:
def diff_min(end: pd.Series, start: pd.Series) -> pd.Series:
    return ((end - start).dt.total_seconds() / 60).round(2)

df_feature['wait_call_min']      = diff_min(df_feature['CarConfirm'],      df_feature['OperatorCarConfirm'])
df_feature['prepare_loading_min'] = diff_min(df_feature['FirstPostPallet'], df_feature['CarConfirm'])
df_feature['loading_time_min']   = diff_min(df_feature['LastPostPallet'],   df_feature['FirstPostPallet'])
df_feature['close_job_min']      = diff_min(df_feature['PostingTime'],      df_feature['LastPostPallet'])

phase_cols = ['wait_call_min', 'prepare_loading_min', 'loading_time_min', 'close_job_min']

print('Null counts:')
print(df_feature[phase_cols].isna().sum())
print()
print('Negative counts:')
print((df_feature[phase_cols] < 0).sum())
print()
df_feature[phase_cols].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T.round(2)


Null counts:
wait_call_min          0
prepare_loading_min    0
loading_time_min       0
close_job_min          0
dtype: int64

Negative counts:
wait_call_min          0
prepare_loading_min    0
loading_time_min       0
close_job_min          0
dtype: int64



,count,mean,std,min,50%,75%,90%,95%,99%,max
wait_call_min,26538.0,8.97,8.95,0.05,6.03,11.62,19.81,26.70,43.98,88.73
prepare_loading_min,26538.0,8.94,10.33,0.10,5.38,12.18,21.91,29.32,46.42,301.08
loading_time_min,26538.0,25.15,19.23,0.02,20.80,34.50,51.73,64.20,86.86,213.98
close_job_min,26538.0,16.78,13.45,1.13,13.45,21.37,31.88,39.95,58.76,339.90


## Save Feature Data

บันทึกผลลัพธ์หลังสร้างฟีเจอร์แล้วไปยังโฟลเดอร์ `data/interim`


## Queue State Features — สภาพคิว ณ เวลาที่รถเข้าโรงงาน

นับจำนวนรถที่ **เข้าก่อนคันปัจจุบัน** (`OperatorCarConfirm[j] < t`) และยังค้างอยู่ในแต่ละ phase ณ เวลา `t`

| Feature | นิยาม | สูตร |
|---|---|---|
| `queue_waiting` | รถที่รออยู่ ยังไม่ CarConfirm | entries_before_t − cc_exits_before_t |
| `queue_loading` | รถที่กำลังโหลด (มี FirstPostPallet ≤ t, ยังไม่มี LastPostPallet) | fp_enters_before_t − lp_enters_before_t |
| `queue_closing` | รถที่โหลดเสร็จ รอปิดงาน (มี LastPostPallet ≤ t, ยังไม่มี PostingTime) | lp_enters_before_t − pt_exits_before_t |
| `total_queue` | รวมรถทุก phase ที่มองเห็น | queue_waiting + queue_loading + queue_closing |
| `available_bays` | ลานโหลดว่างอยู่กี่ลาน | TOTAL_BAYS − queue_loading |

ใช้ **vectorized searchsorted** (O(n log n)) แทน loop เพื่อความเร็ว

In [76]:
import numpy as np

TOTAL_BAYS        = 10  # จำนวนลานโหลดทั้งหมด
QUEUE_CLOSING_CAP = 6   # p99 ของ queue_closing — ค่าเกินนี้คือ data anomaly (batch posting)


def compute_queue_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values('OperatorCarConfirm').reset_index(drop=True)

    for col in ['queue_waiting', 'queue_loading', 'queue_closing']:
        df[col] = 0

    for _, grp in df.groupby('PlantName'):
        idx = grp.index.values

        occ = grp['OperatorCarConfirm'].values
        pt  = grp['PostingTime'].values
        cc  = grp['CarConfirm'].values
        fp  = grp['FirstPostPallet'].values
        lp  = grp['LastPostPallet'].values

        occ_s = np.sort(occ)
        pt_s  = np.sort(pt[~pd.isnull(pt)])
        cc_s  = np.sort(cc[~pd.isnull(cc)])
        fp_s  = np.sort(fp[~pd.isnull(fp)])
        lp_s  = np.sort(lp[~pd.isnull(lp)])

        entries   = np.searchsorted(occ_s, occ, side='left')
        cc_exits  = np.searchsorted(cc_s,  occ, side='right')
        fp_enters = np.searchsorted(fp_s,  occ, side='right')
        lp_enters = np.searchsorted(lp_s,  occ, side='right')
        pt_exits  = np.searchsorted(pt_s,  occ, side='right')

        df.loc[idx, 'queue_waiting'] = entries   - cc_exits
        df.loc[idx, 'queue_loading'] = fp_enters - lp_enters
        df.loc[idx, 'queue_closing'] = lp_enters - pt_exits

    # cap queue_loading ที่ TOTAL_BAYS — ค่าเกินคือ timestamp บันทึกผิด
    df['queue_loading'] = df['queue_loading'].clip(upper=TOTAL_BAYS)

    # cap queue_closing ที่ p99 — ค่าเกินคือ PostingTime batch-enter ผิดปกติ
    df['queue_closing'] = df['queue_closing'].clip(upper=QUEUE_CLOSING_CAP)

    # total_queue = รวมทุก phase ที่มองเห็น (waiting + loading + closing)
    df['total_queue'] = df['queue_waiting'] + df['queue_loading'] + df['queue_closing']

    df['available_bays'] = (TOTAL_BAYS - df['queue_loading']).clip(lower=0)

    return df


df_feature = compute_queue_features(df_feature)

queue_cols = ['total_queue', 'queue_waiting', 'queue_loading', 'queue_closing', 'available_bays']

print('Negative counts:')
print((df_feature[queue_cols] < 0).sum())
print()
print(f'max queue_loading  = {df_feature["queue_loading"].max()}  ← ควร <= {TOTAL_BAYS}')
print(f'max queue_closing  = {df_feature["queue_closing"].max()}  ← ควร <= {QUEUE_CLOSING_CAP}')
print(f'min available_bays = {df_feature["available_bays"].min()}')
print()
print('Correlation กับ total_time_min:')
print(df_feature[queue_cols + ['total_time_min']].corr()['total_time_min'][queue_cols].round(3))
print()
df_feature[queue_cols].describe().round(2)

Negative counts:
total_queue       0
queue_waiting     0
queue_loading     0
queue_closing     0
available_bays    0
dtype: int64

max queue_loading  = 10  ← ควร <= 10
max queue_closing  = 6  ← ควร <= 6
min available_bays = 0

Correlation กับ total_time_min:
total_queue       0.117
queue_waiting     0.112
queue_loading     0.073
queue_closing     0.014
available_bays   -0.073
Name: total_time_min, dtype: float64



,total_queue,queue_waiting,queue_loading,queue_closing,available_bays
count,26538.00,26538.00,26538.00,26538.00,26538.00
mean,5.28,1.16,2.52,1.60,7.48
std,2.75,1.53,1.78,1.39,1.78
min,0.00,0.00,0.00,0.00,0.00
25%,3.00,0.00,1.00,0.00,6.00
50%,5.00,1.00,2.00,1.00,8.00
75%,7.00,2.00,4.00,2.00,9.00
max,20.00,18.00,10.00,6.00,10.00


## Interaction Features

| Feature | สูตร | แนวคิด |
|---|---|---|
| `sap_per_group` | total_sap_amount ÷ product_group_count | ความหนักเฉลี่ยต่อ 1 ประเภทสินค้า |
| `queue_x_bays` | queue_waiting ÷ available_bays | อัตราส่วนแรงกดดันคิว — รอมากลานน้อย = กดดันสูง |
| `queue_x_sap` | queue_waiting × total_sap_amount | คิวยาว + ของเยอะ = ช้าทวีคูณ |

In [77]:
# sap_per_group: ความหนักเฉลี่ยต่อ 1 ประเภทสินค้า (หาร 1 ถ้า product_group_count=0)
df_feature['sap_per_group'] = (
    df_feature['total_sap_amount'] / df_feature['product_group_count'].replace(0, 1)
).round(2)

# queue_x_bays: แรงกดดันคิว (clip ≥ 1 ป้องกันหารศูนย์เมื่อลานเต็ม)
df_feature['queue_x_bays'] = (
    df_feature['queue_waiting'] / df_feature['available_bays'].clip(lower=1)
).round(4)

# queue_x_sap: คิวยาว × ของเยอะ
df_feature['queue_x_sap'] = (
    df_feature['queue_waiting'] * df_feature['total_sap_amount']
).round(2)

interaction_cols = ['sap_per_group', 'queue_x_bays', 'queue_x_sap']

print('Correlation กับ total_time_min:')
corr = df_feature[interaction_cols + ['total_time_min']].corr()['total_time_min'][interaction_cols]
for feat, val in corr.items():
    bar = '█' * int(abs(val) * 20)
    print(f'  {feat:<20}  {val:+.4f}  {"+" if val>=0 else "-"}{bar}')

print()
df_feature[interaction_cols].describe().round(2)

Correlation กับ total_time_min:
  sap_per_group         +0.3286  +██████
  queue_x_bays          +0.1157  +██
  queue_x_sap           +0.3018  +██████



,sap_per_group,queue_x_bays,queue_x_sap
count,26538.00,26538.00,26538.00
mean,1762.06,0.18,2856.87
std,1972.01,0.30,5638.90
min,1.00,0.00,0.00
25%,482.00,0.00,0.00
50%,937.50,0.11,983.00
75%,2195.92,0.25,3400.00
max,12160.00,8.00,141644.00


## Momentum & Interaction Features เพิ่มเติม

| Feature | สูตร | แนวคิด |
|---|---|---|
| `rolling_avg_time_last5` | mean ของ `total_time_min` 5 คันก่อนหน้า (same plant) | momentum — ถ้า 5 คันก่อนนาน คันนี้น่าจะนานด้วย |
| `inter_arrival_min` | `OperatorCarConfirm[i] − OperatorCarConfirm[i−1]` (same plant) | อัตราการมา — รถมาถี่ = congestion สูง |
| `car_type_x_sap` | `CarType × total_sap_amount` | รถใหญ่ + ของมาก = เวลาไม่เพิ่มแบบ linear |

In [78]:
# 1. rolling_avg_time_last5 — mean ของ total_time_min 5 คันก่อนหน้า (same plant)
#    shift(1) ไม่รวมตัวเอง, min_periods=1 ป้องกัน NaN เมื่อคันก่อนหน้าไม่ครบ 5
df_feature = df_feature.sort_values('OperatorCarConfirm').reset_index(drop=True)

df_feature['rolling_avg_time_last5'] = (
    df_feature.groupby('PlantName')['total_time_min']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    .round(2)
)
global_mean = df_feature['total_time_min'].mean()
df_feature['rolling_avg_time_last5'] = df_feature['rolling_avg_time_last5'].fillna(global_mean).round(2)

# 2. inter_arrival_min — นาทีระหว่างรถเข้าติดกัน (same plant)
df_feature['inter_arrival_min'] = (
    df_feature.groupby('PlantName')['OperatorCarConfirm']
    .transform(lambda x: x.diff().dt.total_seconds() / 60)
    .round(2)
)
median_arrival = df_feature['inter_arrival_min'].median()
df_feature['inter_arrival_min'] = df_feature['inter_arrival_min'].fillna(median_arrival).round(2)

# 3. car_type_x_sap — ขนาดรถ × ปริมาณสินค้า (0=4ล้อ, 1=6ล้อ, 2=10ล้อ, 3=เทรเลอร์)
df_feature['car_type_x_sap'] = (df_feature['CarType'] * df_feature['total_sap_amount']).round(2)

new_cols = ['rolling_avg_time_last5', 'inter_arrival_min', 'car_type_x_sap']

print('Correlation กับ total_time_min:')
for feat in new_cols:
    r = df_feature[feat].corr(df_feature['total_time_min'])
    bar = '█' * int(abs(r) * 20)
    print(f'  {feat:<25}  r={r:+.4f}  {"+" if r >= 0 else "-"}{bar}')

print()
df_feature[new_cols].describe().round(2)

Correlation กับ total_time_min:
  rolling_avg_time_last5     r=+0.2769  +█████
  inter_arrival_min          r=+0.0513  +█
  car_type_x_sap             r=+0.5171  +██████████



,rolling_avg_time_last5,inter_arrival_min,car_type_x_sap
count,26538.00,26538.00,26536.00
mean,59.83,27.19,6788.83
std,14.80,274.14,8198.29
min,19.63,0.00,0.00
25%,50.41,2.07,1202.00
50%,58.82,5.60,1650.00
75%,67.93,13.12,14640.00
max,372.46,35647.55,47358.00


## Average Time by CarType Features

| Feature | สูตร | แนวคิด |
|---|---|---|
| `avg_time_by_cartype` | mean ของ `total_time_min` แยกตาม `CarType` | baseline เวลาเฉลี่ยของแต่ละประเภทรถ |
| `rolling_avg_cartype_last10` | mean ของ `total_time_min` 10 คันล่าสุดที่มี `CarType` เดียวกัน | trend เวลาจริงแยกตามประเภทรถ (ไม่มี leakage) |

In [79]:
# 1. avg_time_by_cartype — mean ของ total_time_min แยกตาม CarType (global baseline)
cartype_avg = df_feature.groupby('CarType')['total_time_min'].mean().rename('avg_time_by_cartype')
df_feature = df_feature.join(cartype_avg, on='CarType')
df_feature['avg_time_by_cartype'] = df_feature['avg_time_by_cartype'].round(2)

# 2. rolling_avg_cartype_last10 — mean ของ 10 คันล่าสุดที่มี CarType เดียวกัน
#    df_feature ถูก sort_values('OperatorCarConfirm') ไว้แล้วจาก cell ก่อนหน้า
#    shift(1) ไม่รวมตัวเอง → ไม่มี leakage
df_feature['rolling_avg_cartype_last10'] = (
    df_feature.groupby('CarType')['total_time_min']
    .transform(lambda x: x.shift(1).rolling(window=10, min_periods=1).mean())
    .round(2)
)
# แถวแรกของแต่ละ CarType ที่ยัง NaN → ใช้ avg_time_by_cartype แทน
df_feature['rolling_avg_cartype_last10'] = df_feature['rolling_avg_cartype_last10'].fillna(
    df_feature['avg_time_by_cartype']
)

cartype_cols = ['avg_time_by_cartype', 'rolling_avg_cartype_last10']

print('ค่าเฉลี่ยเวลาแยกตาม CarType:')
print(df_feature.groupby('CarType')[['total_time_min', 'avg_time_by_cartype']].first().round(2))
print()
print('Correlation กับ total_time_min:')
for feat in cartype_cols:
    r = df_feature[feat].corr(df_feature['total_time_min'])
    bar = '█' * int(abs(r) * 20)
    print(f'  {feat:<30}  r={r:+.4f}  {"+" if r >= 0 else "-"}{bar}')

print()
df_feature[cartype_cols].describe().round(2)

ค่าเฉลี่ยเวลาแยกตาม CarType:
         total_time_min  avg_time_by_cartype
CarType                                     
0.0               25.03                38.77
1.0               40.38                52.28
2.0               58.77                59.87
3.0               72.05                74.15

Correlation กับ total_time_min:
  avg_time_by_cartype             r=+0.4572  +█████████
  rolling_avg_cartype_last10      r=+0.4982  +█████████



,avg_time_by_cartype,rolling_avg_cartype_last10
count,26536.00,26536.00
mean,59.83,59.84
std,11.56,16.30
min,38.77,23.52
25%,52.28,47.63
50%,52.28,58.42
75%,74.15,71.10
max,74.15,335.82


In [80]:
output_path = Path('../../data/interim/vw_timestamp_dashboard_featured.csv').resolve()
df_feature.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'Saved → {output_path}')
print(f'Shape  : {df_feature.shape}')

Saved → C:\SCG-Roofing\WMS-ML\WMS-ML\data\interim\vw_timestamp_dashboard_featured.csv
Shape  : (26538, 68)
